Assignment 3 for EN535 
Author: Jeremy Ball

Problem 1 (50 pts)
Create a sphere that moves bank and forth on a line by stopping at the end points and moving with maximum velocity on the mid-point using CoppeliaSim APIs.


Note to grader:
I am using this on Windows 11 with the latest version of CoppeliaSim. On problem 2, if you want to run this script to verify what it does, I put in the path specific location of the model for a standard Windows install, but it could be different on your machine. To run the simulation, it is expected that the simulation is started for moving the sphere. For UR5, we start and stop the simulation.

Problem 4 (35 pts)
Write a computer program that reads a DH table and calculates the forward kinematic equations for a 3-link cylindrical manipulator as shown in Figure 3.7.  Using Coppeliasim, build and program this robot to execute a trajectory of your choosing. The two parts of this problem can be independent (i.e. use Matlab or Python to read in your DH table and generate your forward kinematics. Using CoppeliaSim, build a 3-link cylindrical manipulator based on your DH table and execute a trajectory by commanding the joint angles. 

In [5]:
import time
import numpy as np
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

client = RemoteAPIClient()
sim = client.getObject('sim')

# --- Create helper functions ---

def create_cylinder(radius, height):
    return sim.createPrimitiveShape(sim.primitiveshape_cylinder, [radius, radius, height])

def create_joint(joint_type):
    j = sim.createJoint(joint_type, sim.jointmode_force, 0)
    sim.setObjectFloatParam(j, sim.jointfloatparam_upper_limit, 1.0)
    return j

# --- Build the robot ---

# Base
base = create_cylinder(0.1, 0.05)
sim.setObjectPosition(base, -1, [0, 0, 0.1])

# Joint 1 (revolute)
joint1 = create_joint(sim.joint_revolute_subtype)
sim.setObjectParent(joint1, base, True)

# Link 1
link1 = create_cylinder(0.05, 0.3)
sim.setObjectParent(link1, joint1, True)
sim.setObjectPosition(link1, joint1, [0, 0, 0.15])

# Joint 2 (prismatic)
joint2 = create_joint(sim.joint_prismatic_subtype)
sim.setObjectParent(joint2, link1, True)
sim.setObjectPosition(joint2, link1, [0, 0, 0.3])

# Link 2
link2 = create_cylinder(0.04, 0.3)
sim.setObjectParent(link2, joint2, True)
sim.setObjectPosition(link2, joint2, [0, 0, 0.15])

# Joint 3 (prismatic)
joint3 = create_joint(sim.joint_prismatic_subtype)
sim.setObjectParent(joint3, link2, True)
sim.setObjectPosition(joint3, link2, [0, 0, 0.3])

# Link 3
link3 = create_cylinder(0.03, 0.25)
sim.setObjectParent(link3, joint3, True)
sim.setObjectPosition(link3, joint3, [0, 0, 0.125])

# --- Set joint control modes ---
# All joints in position control
sim.setJointMode(joint1, sim.jointmode_force, 0)
sim.setJointMode(joint2, sim.jointmode_force, 0)
sim.setJointMode(joint3, sim.jointmode_force, 0)

# --- Movement helper ---
def move_to(theta1, d2, d3, duration=2.0):
    start = time.time()
    while time.time() - start < duration:
        sim.setJointTargetPosition(joint1, theta1)
        sim.setJointTargetPosition(joint2, d2)
        sim.setJointTargetPosition(joint3, d3)
        time.sleep(0.01)

# --- Move to 3 poses ---

move_to(theta1=0*np.pi/180,   d2=0.10, d3=0.05)
move_to(theta1=90*np.pi/180,  d2=0.25, d3=0.15)
move_to(theta1=-45*np.pi/180, d2=0.05, d3=0.25)

print("Done.")

Done.


In [ ]:
import time
import numpy as np
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

# ------------------------------------------------------------
# MTB PROGRAM INTERPRETER (Python version)
# ------------------------------------------------------------

class MTBInterpreter:
    def __init__(self, program_text):
        self.lines = [l.strip() for l in program_text.splitlines() if l.strip()]
        self.labels = self._scan_labels()
        self.pc = 0
        self.rot_vel = np.deg2rad(45)
        self.lin_vel = 0.1
        self.output_bits = [0, 0, 0, 0]
        self.input_bits = [0, 0, 0, 0]
        self.joint_targets = [0, 0, 0, 0]
        self.wait_timer = 0

    def _scan_labels(self):
        labels = {}
        for i, line in enumerate(self.lines):
            if not line.startswith(("REM", "SET", "MOVE", "WAIT", "IF", "CLEAR", "GOTO")):
                labels[line] = i
        return labels

    def set_input_bits(self, bits):
        self.input_bits = bits

    def get_output_bits(self):
        return self.output_bits

    def step(self, dt):
        if self.wait_timer > 0:
            self.wait_timer -= dt * 1000
            return self.joint_targets

        if self.pc >= len(self.lines):
            self.pc = 0  # loop forever like Lua version

        line = self.lines[self.pc]

        # ------------------------------------------------------------
        # COMMAND PARSING
        # ------------------------------------------------------------
        if line.startswith("REM"):
            self.pc += 1

        elif line.startswith("SETROTVEL"):
            v = float(line.split()[1])
            self.rot_vel = np.deg2rad(v)
            self.pc += 1

        elif line.startswith("SETLINVEL"):
            v = float(line.split()[1])
            self.lin_vel = v
            self.pc += 1

        elif line.startswith("MOVE"):
            _, p1, p2, p3, p4 = line.split()
            self.joint_targets = [
                np.deg2rad(float(p1)),
                np.deg2rad(float(p2)),
                float(p3),
                np.deg2rad(float(p4)),
            ]
            self.pc += 1

        elif line.startswith("WAIT"):
            ms = float(line.split()[1])
            self.wait_timer = ms
            self.pc += 1

        elif line.startswith("SETBIT"):
            bit = int(line.split()[1])
            self.output_bits[bit // 8] |= (1 << (bit % 8))
            self.pc += 1

        elif line.startswith("CLEARBIT"):
            bit = int(line.split()[1])
            self.output_bits[bit // 8] &= ~(1 << (bit % 8))
            self.pc += 1

        elif line.startswith("IFBITGOTO"):
            _, bit, label = line.split()
            bit = int(bit)
            if self.input_bits[bit // 8] & (1 << (bit % 8)):
                self.pc = self.labels[label]
            else:
                self.pc += 1

        elif line.startswith("IFNBITGOTO"):
            _, bit, label = line.split()
            bit = int(bit)
            if not (self.input_bits[bit // 8] & (1 << (bit % 8))):
                self.pc = self.labels[label]
            else:
                self.pc += 1

        elif line.startswith("GOTO"):
            _, label = line.split()
            self.pc = self.labels[label]

        else:
            # label
            self.pc += 1

        return self.joint_targets


# ------------------------------------------------------------
# PYTHON-COPPELIASIM CONTROL LOOP
# ------------------------------------------------------------

client = RemoteAPIClient()
sim = client.getObject("sim")

# Get robot and joints
robot = sim.getObject('/MTB')
joints = [sim.getObject('/MTB/axis', {'index': i}) for i in range(4)]

# Load the program text (same as Lua)
program_text = """
SETROTVEL 45
SETLINVEL 0.1
MOVE 0 0 0 0
PROGRAM_BEGIN_LABEL
MOVE 0 0 0.03 0
IFBITGOTO 1 LABEL1
SETBIT 1
LABEL1
WAIT 500
MOVE 0 0 0 0
WAIT 250
MOVE -160 -43.5 0 203.5
WAIT 250
MOVE -160 -43.5 0.03 203.5
CLEARBIT 1
WAIT 500
MOVE -160 -43.5 0 203.5
WAIT 250
MOVE 160 43.5 0 -203.5
WAIT 250
MOVE 160 43.5 0.03 -203.5
IFBITGOTO 1 LABEL2
SETBIT 1
LABEL2
WAIT 500
MOVE 160 43.5 0 -203.5
GOTO LABEL1
"""

interp = MTBInterpreter(program_text)

sim.startSimulation()

last_time = time.time()

while sim.getSimulationState() != sim.simulation_stopped:
    now = time.time()
    dt = now - last_time
    last_time = now

    # Read input bits (if you want to simulate them)
    input_bits = [0, 0, 0, 0]
    interp.set_input_bits(input_bits)

    # Run interpreter
    targets = interp.step(dt)

    # Apply joint targets
    for j, t in zip(joints, targets):
        sim.setJointTargetPosition(j, t)

    time.sleep(0.01)

sim.stopSimulation()
